In [ ]:
#| default_exp blast_backend

In [ ]:
#| export
"""Local BLAST specificity with database provenance and retained raw hits."""
from __future__ import annotations

from dataclasses import asdict, dataclass
import json
from pathlib import Path
import subprocess
import sys
import tempfile

from probeharbour.models import Candidate

In [ ]:
#| export
@dataclass(frozen=True)
class BlastHit:
    subject_id: str
    percent_identity: float
    alignment_length: int
    query_start: int
    query_end: int
    subject_start: int
    subject_end: int
    evalue: float
    bit_score: float


@dataclass(frozen=True)
class BlastEvidence:
    candidate_id: str
    target_id: str | None
    database_id: str
    database_version: str
    method_id: str
    command: tuple[str, ...]
    hits: tuple[BlastHit, ...]
    target_raw_score: float
    best_offtarget_raw_score: float
    best_offtarget_id: str | None
    off_target_count: int

    @property
    def specificity_percent(self) -> float:
        if self.target_raw_score <= 0:
            return 0.0
        return round(100 * (1 - min(1.0, self.best_offtarget_raw_score / self.target_raw_score)), 3)

    def as_dict(self) -> dict:
        result = asdict(self)
        result["specificity_percent"] = self.specificity_percent
        return result

In [ ]:
#| export
class LocalBlastBackend:
    """Run ``blastn-short`` against a local BLAST database, with an in-memory cache."""

    def __init__(self, database: str | Path, *, database_id: str, database_version: str,
                 executable: str | Path | None = None, method_id: str = "blastn-short-v1"):
        self.database = Path(database)
        self.database_id = database_id
        self.database_version = database_version
        sibling = Path(sys.executable).with_name("blastn")
        self.executable = str(executable or (sibling if sibling.is_file() else "blastn"))
        self.method_id = method_id
        self._cache: dict[tuple[str, str | None], BlastEvidence] = {}

    def evaluate(self, candidate: Candidate) -> tuple[BlastEvidence, bool]:
        key = (candidate.binder, candidate.transcript_id)
        if key in self._cache:
            return self._cache[key], True
        evidence = self._run(candidate)
        self._cache[key] = evidence
        return evidence, False

    def _run(self, candidate: Candidate) -> BlastEvidence:
        fields = "sseqid pident length qstart qend sstart send evalue bitscore"
        command = (
            self.executable, "-task", "blastn-short", "-db", str(self.database),
            "-query", "QUERY_FASTA", "-outfmt", f"6 {fields}", "-max_target_seqs", "100",
        )
        with tempfile.TemporaryDirectory(prefix="probestore-blast-") as directory:
            query = Path(directory) / "query.fasta"
            query.write_text(f">{candidate.candidate_id}\n{candidate.binder}\n")
            actual_command = [str(query) if value == "QUERY_FASTA" else value for value in command]
            completed = subprocess.run(actual_command, check=True, capture_output=True, text=True)
        hits = tuple(_parse_hit(line) for line in completed.stdout.splitlines() if line.strip())
        target_hits = [hit for hit in hits if candidate.transcript_id and candidate.transcript_id in hit.subject_id]
        off_hits = [hit for hit in hits if not candidate.transcript_id or candidate.transcript_id not in hit.subject_id]
        target_score = max((hit.bit_score for hit in target_hits), default=0.0)
        best_offtarget = max(off_hits, key=lambda hit: hit.bit_score, default=None)
        return BlastEvidence(
            candidate.candidate_id, candidate.transcript_id, self.database_id, self.database_version,
            self.method_id, command, hits, target_score,
            best_offtarget.bit_score if best_offtarget else 0.0,
            best_offtarget.subject_id if best_offtarget else None, len(off_hits),
        )

In [ ]:
#| export
def check_specificity(store, backend: LocalBlastBackend, *, candidate_ids=None):
    """Annotate a ProbeStore and retain complete JSON-safe BLAST evidence."""
    wanted = set(candidate_ids) if candidate_ids is not None else None
    records = []
    for candidate in store.to_candidates():
        if wanted is not None and candidate.candidate_id not in wanted:
            continue
        evidence, cache_hit = backend.evaluate(candidate)
        record = evidence.as_dict()
        record["cache_hit"] = cache_hit
        records.append(record)
    result = {record["candidate_id"]: record for record in records}
    ids = store._candidate_ids()
    store.obs["specificity_percent_recalculated"] = ids.map(lambda value: result.get(value, {}).get("specificity_percent"))
    store.obs["specificity_evidence"] = ids.map(lambda value: json.dumps(result[value], sort_keys=True) if value in result else None)
    return store

In [ ]:
#| export
def _parse_hit(line: str) -> BlastHit:
    values = line.split("\t")
    return BlastHit(values[0], float(values[1]), int(values[2]), int(values[3]), int(values[4]),
                    int(values[5]), int(values[6]), float(values[7]), float(values[8]))